# Notebook 02 — Model Training

Runs `main.py --train` and streams all terminal output directly into this cell.
Downstream cells load results from saved files so they can be re-run independently.

**Pre-requisites:**
- `.venv` activated: `source .venv/bin/activate && jupyter notebook`
- Custom operator images in `data/symbols/` (optional — digits-only training works without them)

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import config

print('Project root :', PROJECT_ROOT)
print('Classes      :', config.NUM_CLASSES)
print('Image size   :', config.IMAGE_SIZE)
print('Epochs       :', config.EPOCHS)
print('Batch size   :', config.BATCH_SIZE)
print('LR           :', config.LEARNING_RATE)

## 1. Train — streams full terminal output here

In [ ]:
import subprocess

proc = subprocess.Popen(
    [sys.executable, str(PROJECT_ROOT / 'main.py'), '--train'],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr into stdout
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\n[Done] exit code: {proc.returncode}')

## 2. Training curves  *(loads from saved CSV — no retraining needed)*

In [ ]:
import pandas as pd

log_path = PROJECT_ROOT / 'models' / 'training_log.csv'
if not log_path.exists():
    print('training_log.csv not found — run the training cell first.')
else:
    df = pd.read_csv(log_path)
    print(df.tail())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df['epoch'], df['accuracy'],     label='Train')
    axes[0].plot(df['epoch'], df['val_accuracy'], label='Val')
    axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(df['epoch'], df['loss'],     label='Train')
    axes[1].plot(df['epoch'], df['val_loss'], label='Val')
    axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = PROJECT_ROOT / 'output' / 'reports' / '02_training_curves.png'
    plt.savefig(save_path, dpi=120)
    plt.show()
    print(f'Best val accuracy: {df["val_accuracy"].max():.4f}')

## 3. Confusion matrix  *(loads model + test split from disk)*

In [ ]:
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report

if not config.MODEL_PATH.exists():
    print('Model not found — run the training cell first.')
else:
    model = tf.keras.models.load_model(str(config.MODEL_PATH))

    proc_dir = PROJECT_ROOT / 'data' / 'processed'
    X_test = np.load(proc_dir / 'X_test.npy')
    y_test = np.load(proc_dir / 'y_test.npy')

    # Add channel dim if missing
    if X_test.ndim == 3:
        X_test = X_test[..., np.newaxis]
    # One-hot decode if needed
    y_true = y_test.argmax(axis=1) if y_test.ndim == 2 else y_test

    y_pred  = model.predict(X_test, verbose=1).argmax(axis=1)
    present = sorted(np.unique(np.concatenate([y_true, y_pred])))
    labels  = [config.CLASS_MAP[i] for i in present]

    cm = confusion_matrix(y_true, y_pred, labels=present)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — Test Set')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'output' / 'reports' / '02_confusion_matrix.png', dpi=120)
    plt.show()

    print('\nClassification report:')
    print(classification_report(y_true, y_pred, labels=present, target_names=labels))

## 4. Grad-CAM on random test samples

In [ ]:
if not config.MODEL_PATH.exists():
    print('Model not found — run the training cell first.')
else:
    from src.gradcam import GradCAM

    gcam   = GradCAM(model)
    n_show = 5
    idxs   = np.random.choice(len(X_test), n_show, replace=False)

    fig, axes = plt.subplots(n_show, 3, figsize=(9, n_show * 2.5))
    fig.suptitle('Grad-CAM — Test Samples', fontsize=12)

    for row, idx in enumerate(idxs):
        img    = X_test[idx]
        true_c = int(y_true[idx])
        pred_c = int(model.predict(img[None], verbose=0).argmax())
        heatmap, overlay = gcam.compute_gradcam(img, pred_c)

        axes[row, 0].imshow(img.squeeze(), cmap='gray')
        axes[row, 0].set_title(f'Input  true={config.CLASS_MAP[true_c]}', fontsize=8)
        axes[row, 1].imshow(heatmap)
        axes[row, 1].set_title('Grad-CAM', fontsize=8)
        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title(f'Overlay pred={config.CLASS_MAP[pred_c]}', fontsize=8)
        for ax in axes[row]:
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'output' / 'reports' / '02_gradcam_samples.png', dpi=120)
    plt.show()